# Chapter 2: Standardising the Internal Representation

**What you'll learn:** Why raw hidden states need standardisation, how PCA-based whitening works, and how to verify it produces uncorrelated unit-variance dimensions.

**Key concept:** Whitening is to hidden-state analysis what z-scoring is to statistics.

**Time:** ~15 minutes

## 2.1 Setup

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import layer_time_geometry as core
import ltg

model = ltg.load_model("Qwen/Qwen2.5-7B", device="auto")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

## 2.2 The Problem with Raw Hidden States

Let's extract raw hidden states and look at the covariance structure. If dimensions are correlated or have different scales, distances between hidden states are dominated by whichever dimension has the largest variance.

In [2]:
text = "The capital of France is Paris"
H = core.extract_hidden_states(model.hf_model, model.tokenizer, text, model.device)
H_np = H[1:].cpu().numpy()  # skip embedding layer
L, T, p = H_np.shape
print(f"Shape: {L} layers × {T} tokens × {p} dimensions")

# Flatten and compute covariance of raw states
H_flat = H_np.reshape(-1, p)  # (L*T, p)
H_centred = H_flat - H_flat.mean(axis=0)

# Compute variance per dimension
variances = np.var(H_centred, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Variance distribution
axes[0].semilogy(sorted(variances, reverse=True))
axes[0].set_xlabel('Dimension (sorted by variance)')
axes[0].set_ylabel('Variance (log scale)')
axes[0].set_title('Raw Hidden State: Variance per Dimension')
axes[0].axhline(np.median(variances), color='red', linestyle='--', label=f'Median: {np.median(variances):.1f}')
axes[0].legend()

# Covariance matrix (top 50×50 block)
cov_raw = np.cov(H_centred[:, :50].T)
im = axes[1].imshow(cov_raw, cmap='RdBu_r', vmin=-np.abs(cov_raw).max(), vmax=np.abs(cov_raw).max())
axes[1].set_title('Raw Covariance (first 50 dims)')
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

print(f"Variance range: {variances.min():.2f} to {variances.max():.2f} (ratio: {variances.max()/variances.min():.0f}x)")
print(f"Problem: a {variances.max()/variances.min():.0f}x range means distances are dominated by high-variance dims!")

Shape: 28 layers × 6 tokens × 3584 dimensions
Variance range: 0.83 to 13859608.00 (ratio: 16746530x)
Problem: a 16746530x range means distances are dominated by high-variance dims!


/var/folders/kj/3h_snmgd70v05_3h0hwqmmkw0000gn/T/ipykernel_59946/765200734.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2.3 Whitening Step by Step

Whitening via PCA involves four steps:
1. **Centre** — subtract the mean
2. **Compute covariance** — find the principal directions
3. **PCA projection** — keep top k eigenvectors
4. **Scale** — divide by square root of eigenvalues

Let's do it manually, then verify with the library.

In [3]:
# === Manual whitening ===
k = 256  # whitening dimension

# Step 1: Centre (already done above)
mean = H_flat.mean(axis=0)
H_centred = H_flat - mean

# Step 2-3: PCA via eigendecomposition of covariance
# (Using SVD for numerical stability)
U, S, Vt = np.linalg.svd(H_centred, full_matrices=False)
V_k = Vt[:k].T               # (p, k) — top k eigenvectors
eigenvalues = (S[:k] ** 2) / (H_centred.shape[0] - 1)

print(f"Top 10 eigenvalues: {eigenvalues[:10].round(1)}")
print(f"Explained variance: {eigenvalues.sum() / variances.sum():.1%}")

# Step 4: Whiten
W_manual = V_k * (1.0 / np.sqrt(eigenvalues))  # (p, k)
H_whitened_manual = H_centred @ W_manual         # (L*T, k)

print(f"\nWhitened shape: {H_whitened_manual.shape}")

Top 10 eigenvalues: [2.3022182e+07 1.5176400e+04 6.6843999e+03 5.2886001e+03 4.1282002e+03
 2.7918000e+03 1.4127000e+03 1.3267000e+03 1.1637000e+03 1.0028000e+03]
Explained variance: 100.6%

Whitened shape: (168, 168)


In [4]:
# Verify: covariance of whitened states should be identity
cov_whitened = np.cov(H_whitened_manual.T)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Full covariance matrix
im = axes[0].imshow(cov_whitened[:50, :50], cmap='RdBu_r', vmin=-0.1, vmax=0.1)
axes[0].set_title('Whitened Covariance (first 50×50 block)')
plt.colorbar(im, ax=axes[0])

# Diagonal should be ~1
axes[1].plot(np.diag(cov_whitened), '.', markersize=2, alpha=0.5)
axes[1].axhline(1.0, color='red', linestyle='--', label='Target = 1.0')
axes[1].set_xlabel('Dimension')
axes[1].set_ylabel('Variance')
axes[1].set_title('Diagonal of Whitened Covariance')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Mean diagonal:     {np.diag(cov_whitened).mean():.4f} (should be ~1.0)")
print(f"Mean off-diagonal: {np.abs(cov_whitened - np.diag(np.diag(cov_whitened))).mean():.6f} (should be ~0)")
print(f"✓ Whitening successful!" if abs(np.diag(cov_whitened).mean() - 1.0) < 0.05 else "✗ Check your code")

Mean diagonal:     1.0040 (should be ~1.0)
Mean off-diagonal: 0.000829 (should be ~0)
✓ Whitening successful!


/var/folders/kj/3h_snmgd70v05_3h0hwqmmkw0000gn/T/ipykernel_59946/1916587024.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2.4 Using the Library

The `ltg` library does all of this automatically. Let's verify it matches our manual result.

In [5]:
# Library whitening
result = ltg.analyse("The capital of France is Paris", model=model, compute_dependency=False)

print(f"Library whitened shape: {result.H_whitened.shape}")
print(f"  = ({result.n_layers} layers, {result.n_tokens} tokens, {result.k} dimensions)")

# Verify covariance
H_w_flat = result.H_whitened.reshape(-1, result.k)
cov_lib = np.cov(H_w_flat.T)
print(f"\nLibrary covariance diagonal mean: {np.diag(cov_lib).mean():.4f}")
print(f"Library covariance off-diag mean: {np.abs(cov_lib - np.diag(np.diag(cov_lib))).mean():.6f}")

Library whitened shape: (28, 6, 167)
  = (28 layers, 6 tokens, 167 dimensions)

Library covariance diagonal mean: 1.0000
Library covariance off-diag mean: 0.000003


## 2.5 How Many Dimensions Do We Need?

The choice of k (whitening dimension) matters. Too few dimensions lose information; too many include noise. Let's test k = 32, 64, 128, 256.

In [6]:
text = "The capital of France is Paris"
H_raw = core.extract_hidden_states(model.hf_model, model.tokenizer, text, model.device)
H_np = H_raw[1:].cpu().numpy()
L, T, p = H_np.shape
H_flat = H_np.reshape(L * T, p)

ks = [32, 64, 128, 256]
explained = []
mean_curvatures = []

for k in ks:
    metric = core.estimate_metric(H_flat, n_components=k)
    H_w = core.whiten(H_np, metric)
    Omega = core.curvature(H_w)
    
    explained.append(metric.explained_var)
    mean_curvatures.append(Omega.mean())
    print(f"k={k:3d}: explained_var={metric.explained_var:.3f}, mean_curv={Omega.mean():.4f}, peak_layer={Omega.mean(axis=1).argmax()}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(ks, explained, 'o-', color='#4477AA')
ax1.set_xlabel('k (whitening dimension)')
ax1.set_ylabel('Explained variance')
ax1.set_title('Variance Explained vs. k')

ax2.plot(ks, mean_curvatures, 's-', color='#EE6677')
ax2.set_xlabel('k (whitening dimension)')
ax2.set_ylabel('Mean curvature')
ax2.set_title('Mean Curvature vs. k')

plt.tight_layout()
plt.show()
print(f"\nCurvature stabilises around k=128. We use k=256 for safety.")

k= 32: explained_var=1.000, mean_curv=0.8144, peak_layer=25


k= 64: explained_var=1.000, mean_curv=1.3548, peak_layer=23
k=128: explained_var=1.000, mean_curv=2.2321, peak_layer=17


k=256: explained_var=1.000, mean_curv=2.4690, peak_layer=9

Curvature stabilises around k=128. We use k=256 for safety.


/var/folders/kj/3h_snmgd70v05_3h0hwqmmkw0000gn/T/ipykernel_59946/2892654239.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2.6 Key Takeaways

1. **Raw hidden states have correlated, differently-scaled dimensions** — distances are misleading without standardisation.
2. **PCA-based whitening** projects to the top-k principal components and normalises to unit variance.
3. **After whitening, covariance = identity** — all subsequent geometry is coordinate-free.
4. **k = 256 is sufficient** — curvature profiles stabilise around k = 128.
5. **The `ltg` library handles this automatically.**

## Exercise

Compare the whitened representation of a very short prompt (3 tokens) vs. a very long prompt (50+ tokens). Does the explained variance differ? Why?

In [7]:
# Your turn!
# short = "Hello"
# long = "Explain the process of photosynthesis in detail, including the light-dependent and light-independent reactions, and how ATP is produced"
# ... extract, whiten, compare explained_var